In [19]:
import PyPDF2
from PyPDF2 import PdfReader

def is_header(line):
    headers = ['TÍTULO', 'CAPÍTULO', 'SECÇÃO', 'SUBSECÇÃO']
    line_upper = line.upper()
    return any(line_upper.startswith(h) for h in headers)

def extract_articles(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    
    # Clean hyphens and handle newlines
    text = text.replace('-\n', '')
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    current_hierarchy = {
        'title': None,
        'chapter': None,
        'section': None,
        'subsection': None
    }
    articles = []
    i = 0
    
    while i < len(lines):
        line = lines[i]
        line_upper = line #.upper()
        
        if line_upper.startswith('TÍTULO'):
            title_header = line
            title_desc = ''
            if i + 1 < len(lines):
                next_line = lines[i + 1]
                if not is_header(next_line) and not next_line.startswith('Artigo'):
                    title_desc = next_line
                    i += 1
            current_hierarchy['title'] = (title_header, title_desc)
            current_hierarchy['chapter'] = None  # Reset lower hierarchies
            current_hierarchy['section'] = None
            current_hierarchy['subsection'] = None
            i += 1
        
        elif line_upper.startswith('CAPÍTULO'):
            chapter_header = line
            chapter_desc = ''
            if i + 1 < len(lines):
                next_line = lines[i + 1]
                if not is_header(next_line) and not next_line.startswith('Artigo'):
                    chapter_desc = next_line
                    i += 1
            current_hierarchy['chapter'] = (chapter_header, chapter_desc)
            current_hierarchy['section'] = None  # Reset lower hierarchies
            current_hierarchy['subsection'] = None
            i += 1
        
        elif line_upper.startswith('SECÇÃO'):
            section_header = line
            section_desc = ''
            if i + 1 < len(lines):
                next_line = lines[i + 1]
                if not is_header(next_line) and not next_line.startswith('Artigo'):
                    section_desc = next_line
                    i += 1
            current_hierarchy['section'] = (section_header, section_desc)
            current_hierarchy['subsection'] = None  # Reset lower hierarchy
            i += 1
        
        elif line_upper.startswith('SUBSECÇÃO'):
            subsection_header = line
            subsection_desc = ''
            if i + 1 < len(lines):
                next_line = lines[i + 1]
                if not is_header(next_line) and not next_line.startswith('Artigo'):
                    subsection_desc = next_line
                    i += 1
            current_hierarchy['subsection'] = (subsection_header, subsection_desc)
            i += 1
        
        elif line.startswith('Artigo'):
            article_lines = [line]
            j = i + 1
            while j < len(lines):
                next_line = lines[j]
                if is_header(next_line) or next_line.startswith('Artigo'):
                    break
                article_lines.append(next_line)
                j += 1
            article_text = ' '.join(article_lines).strip()
            articles.append({
                'hierarchy': {
                    'title': current_hierarchy['title'],
                    'chapter': current_hierarchy['chapter'],
                    'section': current_hierarchy['section'],
                    'subsection': current_hierarchy['subsection']
                },
                'text': article_text
            })
            i = j
        
        else:
            i += 1
    
    return articles

def write_articles_to_file(articles, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        for article in articles:
            hierarchy = article['hierarchy']
            output_lines = []
            
            # Always include Title and description if present
            if hierarchy['title']:
                output_lines.append(hierarchy['title'][0])
                if hierarchy['title'][1]:
                    output_lines.append(hierarchy['title'][1])
            
            # Always include Chapter and description if present
            if hierarchy['chapter']:
                output_lines.append(hierarchy['chapter'][0])
                if hierarchy['chapter'][1]:
                    output_lines.append(hierarchy['chapter'][1])
            
            # Always include Section and description if present
            if hierarchy['section']:
                output_lines.append(hierarchy['section'][0])
                if hierarchy['section'][1]:
                    output_lines.append(hierarchy['section'][1])
            
            # Always include Subsection and description if present
            if hierarchy['subsection']:
                output_lines.append(hierarchy['subsection'][0])
                if hierarchy['subsection'][1]:
                    output_lines.append(hierarchy['subsection'][1])
            
            # Add article text
            output_lines.append(article['text'])
            
            # Write to file
            f.write('\n'.join(output_lines) + '\n\n')

if __name__ == '__main__':
    articles = extract_articles('PDM_Porto_Aviso n.º 1934_2023.pdf')
    write_articles_to_file(articles, 'extracted_articles.txt')

In [20]:
import json

articles
with open('articles.json', 'w', encoding='utf-8') as json_file:
    json.dump(articles, json_file, ensure_ascii=False, indent=4)